# WAF Assessment — Run All Pillars

Runs every pillar in succession, then combines the results into one final report you can
export.

**This is the only notebook you need to run.** It loads `00_config`, executes each
pillar's checks, and produces:

- a **markdown report** (per-pillar rollup, priority findings, full results)
- a **CSV export** for spreadsheets
- the **results table**, queryable in SQL

Configure `00_config` first — this notebook stops immediately if `CATALOG`/`SCHEMA` are unset.

In [ ]:
%run ./00_config

## 1. Choose pillars

All six implemented pillars run by default. Comment out any you want to skip.

**Security, Compliance & Privacy (38 questions) is not implemented yet.** Its questions
are excluded from this run and from the score; assess that pillar manually in the WAF
Assessment Tool.

In [ ]:
PILLARS_TO_RUN = [
    ("checks.governance",             "data-ai-governance"),
    ("checks.interoperability",       "interoperability-usability"),
    ("checks.operational_excellence", "operational-excellence"),
    ("checks.reliability",            "reliability"),
    ("checks.performance",            "performance-efficiency"),
    ("checks.cost",                   "cost-optimization"),
    # ("checks.security",             "security-compliance-privacy"),  # not implemented
]

NOT_IMPLEMENTED = ["security-compliance-privacy"]

total_questions = sum(len(wq.pillar_meta(pid)) for _, pid in PILLARS_TO_RUN)
skipped = sum(len(wq.pillar_meta(pid)) for pid in NOT_IMPLEMENTED)

print(f"Pillars to run   : {len(PILLARS_TO_RUN)}")
print(f"Questions        : {total_questions} of {len(wq.QUESTIONS)} in the WAF question bank")
if skipped:
    print(f"Not implemented  : {skipped} question(s) in "
          f"{', '.join(wq.PILLAR_NAMES[p] for p in NOT_IMPLEMENTED)}")

## 2. Run every pillar

Each pillar is independent: one failing pillar does not stop the rest, and individual
check failures are reported as `open` with the error in the rationale rather than
aborting the run.

In [ ]:
import time
import traceback

all_results = []
pillar_timings = {}
pillar_errors = {}

for module_name, pillar_id in PILLARS_TO_RUN:
    pillar_name = wq.PILLAR_NAMES[pillar_id]
    print("=" * 100)
    print(f"  {pillar_name}")
    print("=" * 100)
    t0 = time.time()
    try:
        results = run_pillar(module_name, pillar_id)
        all_results.extend(results)
    except Exception as exc:
        pillar_errors[pillar_name] = f"{type(exc).__name__}: {exc}"
        print(f"\n!! {pillar_name} FAILED: {type(exc).__name__}: {exc}")
        if CFG.get("debug"):
            traceback.print_exc()
    pillar_timings[pillar_name] = time.time() - t0
    print(f"\n({pillar_timings[pillar_name]:.1f}s)\n")

print("=" * 100)
print(f"Completed {len(all_results)} question(s) across "
      f"{len(PILLARS_TO_RUN) - len(pillar_errors)} pillar(s) in "
      f"{sum(pillar_timings.values()):.1f}s")
if pillar_errors:
    print(f"\n{len(pillar_errors)} pillar(s) failed:")
    for name, err in pillar_errors.items():
        print(f"  - {name}: {err}")

## 3. Combined report

Built from the results table when persistence is on, so a pillar you ran earlier in a
separate notebook is still included. Falls back to this session's in-memory results
otherwise.

In [ ]:
rows = None
if CFG["persist_results"]:
    fqn = wc.results_table_fqn(CFG)
    try:
        rows = [
            r.asDict() for r in
            spark.sql(
                f"""
                SELECT pillar, question_id, principle, question, status, reason
                FROM {fqn}
                WHERE run_id = '{ctx.run_id}'
                ORDER BY pillar, question_id
                """
            ).collect()
        ]
        print(f"Loaded {len(rows)} result(s) from {fqn} for run_id={ctx.run_id}")
    except Exception as exc:
        print(f"Could not read {fqn} ({exc}); using in-memory results instead.")

if not rows:
    rows = [
        {"pillar": r.pillar, "question_id": r.qid, "principle": r.principle,
         "question": r.title, "status": r.status, "reason": r.reason}
        for r in all_results
    ]
    print(f"Using {len(rows)} in-memory result(s)")

workspace_url = ""
try:
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl", "")
except Exception:
    pass

final_report = wc.render_final_report(
    rows,
    run_id=ctx.run_id,
    scope_label=ctx.scope.label,
    workspace=workspace_url,
)
print(f"Report built: {len(final_report):,} characters")

### Rendered report

In [ ]:
# Render the combined report inline; fall back to plain text outside a notebook.
try:
    displayHTML(
        "<div style='max-width:1100px;font-family:system-ui,-apple-system,sans-serif'>"
        "<pre style='white-space:pre-wrap;font-size:12px;line-height:1.5'>"
        + final_report.replace("&", "&amp;").replace("<", "&lt;")
        + "</pre></div>"
    )
except NameError:
    print(final_report)

## 4. Executive summary

In [ ]:
counts = wc.summarize(rows)
scored = counts["open"] + counts["in-progress"] + counts["completed"]

print("=" * 78)
print(f"  WAF CURRENT STATE ASSESSMENT - {ctx.run_id}")
print("=" * 78)
print(f"  Scope     : {ctx.scope.label}")
print(f"  Lookback  : {ctx.lookback_days} days")
print(f"  Questions : {len(rows)}")
print("-" * 78)
for status in ("completed", "in-progress", "open", "manual-review"):
    n = counts[status]
    bar = "#" * int(n / max(len(rows), 1) * 40)
    print(f"  {status:<14} {n:>4}  {bar}")
print("-" * 78)
if scored:
    print(f"  Automated maturity score: {counts['completed']}/{scored} "
          f"({counts['completed'] / scored:.1%})")
    print("  (manual-review questions excluded - they need a human answer)")
print("=" * 78)

# Per-pillar breakdown
by_pillar = {}
for r in rows:
    by_pillar.setdefault(r["pillar"], []).append(r)

print(f"\n{'Pillar':<34}{'Qs':>4}{'Done':>6}{'Prog':>6}{'Open':>6}{'Man':>5}{'Score':>8}")
print("-" * 78)
for pillar, prs in sorted(by_pillar.items()):
    c = wc.summarize(prs)
    s = c["open"] + c["in-progress"] + c["completed"]
    score = f"{c['completed'] / s:.0%}" if s else "n/a"
    print(f"{pillar[:33]:<34}{len(prs):>4}{c['completed']:>6}{c['in-progress']:>6}"
          f"{c['open']:>6}{c['manual-review']:>5}{score:>8}")

## 5. Priority findings

`open` first (no evidence of the control), then `in-progress` (partially adopted).
`manual-review` is excluded here — those need an interview, not remediation.

In [ ]:
actionable = [r for r in rows if r["status"] in ("open", "in-progress")]
actionable.sort(key=lambda r: (wc.STATUS_SORT[r["status"]], r["question_id"]))

print(f"{len(actionable)} actionable finding(s)\n")
for r in actionable:
    print(f"[{r['status']:<11}] {r['question_id']}  {r['question']}")
    print(f"                {r['reason']}\n")

## 6. Questions needing a human answer

These cannot be derived from telemetry. Each rationale names the evidence to gather —
use this as the interview checklist, then record the answers in the WAF Assessment Tool.

In [ ]:
manual = [r for r in rows if r["status"] == "manual-review"]
print(f"{len(manual)} question(s) require a manual answer\n")
for r in manual:
    print(f"{r['question_id']}  {r['question']}")
    print(f"    {r['reason']}\n")

## 7. Export

Three outputs. The markdown file is the one to share; the CSV suits spreadsheets; the
table is there for SQL.

In [ ]:
import os

EXPORT_DIR = f"/Volumes/{CFG['catalog']}/{CFG['schema']}/waf_exports"
USE_VOLUME = False   # set True if you have created the Volume above

md_name = f"waf_assessment_{ctx.run_id}.md"
csv_name = f"waf_assessment_{ctx.run_id}.csv"
csv_text = wc.render_csv(rows)

if USE_VOLUME:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    md_path, csv_path = f"{EXPORT_DIR}/{md_name}", f"{EXPORT_DIR}/{csv_name}"
    with open(md_path, "w") as fh:
        fh.write(final_report)
    with open(csv_path, "w") as fh:
        fh.write(csv_text)
else:
    # /tmp on the driver: readable via dbutils and downloadable from the UI.
    md_path, csv_path = f"/tmp/{md_name}", f"/tmp/{csv_name}"
    with open(md_path, "w") as fh:
        fh.write(final_report)
    with open(csv_path, "w") as fh:
        fh.write(csv_text)

print(f"Markdown : {md_path}")
print(f"CSV      : {csv_path}")
if CFG["persist_results"]:
    print(f"Table    : {wc.results_table_fqn(CFG)}  (run_id = '{ctx.run_id}')")
print(
    "\nTo copy an export into a UC Volume:\n"
    f"  dbutils.fs.cp('file:{md_path}', "
    "'dbfs:/Volumes/<catalog>/<schema>/<volume>/')"
)

### Query results in SQL

```sql
-- Current state by pillar
SELECT pillar, status, count(*) AS questions
FROM <catalog>.<schema>.waf_assessment_results
WHERE run_id = '<run_id>'
GROUP BY pillar, status
ORDER BY pillar, status;

-- Everything still open, worst first
SELECT question_id, pillar, question, reason
FROM <catalog>.<schema>.waf_assessment_results
WHERE run_id = '<run_id>' AND status = 'open'
ORDER BY pillar, question_id;

-- Track a question's status across runs
SELECT run_id, run_timestamp, status, reason
FROM <catalog>.<schema>.waf_assessment_results
WHERE question_id = 'DG-01-02'
ORDER BY run_timestamp DESC;
```